# Causal effects with g-computation

`bambi.causal.g_computation` estimates causal effects from a fitted Bambi model by
*standardization*, also known as g-computation. For each posterior draw, we predict the
expected outcome for every row of a reference population under each intervention,
average those predictions, and then compare the averages. This gives a posterior
distribution for quantities such as the average treatment effect (ATE), the effect among
the treated (ATT) or untreated (ATU), conditional (CATE) and group (GATE) effects, and
dose-response curves.

A regression model only answers a causal question under assumptions that the data cannot
check: no unmeasured confounding (exchangeability), overlap at the intervention values
(positivity), well-defined interventions (consistency), and an adequate outcome model.
`g_computation` asks you to state the treatment, the adjustment set and their temporal
order, and checks that they agree with the model. It does not verify the assumptions.

In [ ]:
import arviz as az
import bambi as bmb
import numpy as np
import pandas as pd
from scipy.special import expit

from bambi.causal import g_computation, plot_effects

## A continuous outcome

We simulate 128 people with a pretreatment covariate `x`, a binary stratum `z`, and one of
four sites. The exposure `dose` takes the values 0, 0.5 and 1, and its effect on `y` is
larger when `z = 1`.

In [ ]:
rng = np.random.default_rng(1055)

n = 128
site = pd.Categorical(np.tile(["north", "south", "east", "west"], n // 4))
x = rng.normal(size=n)
z = rng.integers(0, 2, size=n)
dose = rng.choice([0.0, 0.5, 1.0], size=n, p=[0.35, 0.3, 0.35])
site_effect = np.array([-0.3, 0.2, 0.4, -0.2])[site.codes]
y = 0.5 + (1 + 1.2 * z) * dose + 0.6 * x + site_effect + rng.normal(0, 0.6, n)
data = pd.DataFrame({"y": y, "dose": dose, "x": x, "z": z, "site": site})
data.head()

In [ ]:
model = bmb.Model("y ~ dose * z + x + (1 | site)", data)
idata = model.fit(target_accept=0.95, random_seed=105501)

In [ ]:
az.summary(idata, var_names=["dose", "z", "dose:z", "x", "sigma"])

## Declaring the causal question

Besides the model and its posterior, `g_computation` needs:

* `treatment` and `outcome`;
* `adjustment_set`: every other covariate in the model, including group factors;
* `temporal_order`: the adjustment variables, the treatment and the outcome, in that order;
* `estimand` and the intervention `values`;
* `reference_population`: the rows we average over.

Here the reference population is the observed sample, with equal weights. It is held
fixed, so the intervals below describe posterior uncertainty given these 128 people, not
uncertainty about how they were sampled. We store the shared arguments once.

In [ ]:
causal_inputs = {
    "treatment": "dose",
    "outcome": "y",
    "adjustment_set": ["x", "z", "site"],
    "temporal_order": ["x", "z", "site", "dose", "y"],
    "reference_population": data[["dose", "x", "z", "site"]],
}

## Standardized means and the ATE

Write $m_s(a, l) = E[Y \mid A = a, L = l, \theta_s]$ for the expected outcome in draw $s$.
The standardized mean under intervention $a$ is $M_s(a) = \sum_i w_i \, m_s(a, l_i)$,
where the $l_i$ are the reference rows and the weights $w_i$ sum to one. The ATE is
$M_s(1) - M_s(0)$, computed within each draw and then summarized.

In [ ]:
standardized = g_computation(
    model, idata, estimand="standardized_mean", values=[0.0, 1.0], **causal_inputs
)
standardized.summary

In [ ]:
ate = g_computation(model, idata, estimand="ate", values=[0.0, 1.0], **causal_inputs)
ate.summary

The `summary` is a data frame. The draws behind it are in `ate.draws["effects"]`, with
the original `chain` and `draw` dimensions, so you can compute other summaries.

## ATT and ATU

The ATT averages the same contrast over the people who were originally treated, and the
ATU over those who were not. Because `dose` has three values, we pass the target
population explicitly with `membership`, a boolean mask chosen *before* the
intervention. Here we compare dose 1 with dose 0 among people originally at dose 1 (ATT)
and at dose 0 (ATU).

In [ ]:
att = g_computation(
    model, idata, estimand="att", values=[0.0, 1.0],
    membership=data["dose"].to_numpy() == 1.0, **causal_inputs
)
att.summary

In [ ]:
atu = g_computation(
    model, idata, estimand="atu", values=[0.0, 1.0],
    membership=data["dose"].to_numpy() == 0.0, **causal_inputs
)
atu.summary

## Conditional and group effects

The CATE for `z = 0` and `z = 1` uses the reference rows observed at each value of `z`,
keeping their own distribution of `x` and site. It does not set `z` to a value while
averaging over everyone else, which would answer a different question when covariates are
related. The GATE reports one effect per observed group, here per site and stratum.

In [ ]:
cate = g_computation(
    model, idata, estimand="cate", values=[0.0, 1.0], conditional={"z": [0, 1]},
    **causal_inputs
)
plot_effects(cate, main="z")

In [ ]:
gate = g_computation(
    model, idata, estimand="gate", values=[0.0, 1.0], average_by=["z", "site"],
    **causal_inputs
)
plot_effects(gate, main="site", panel="z")

The effect is larger when `z = 1`, as in the simulation. Group effects of the sites are
included by default; `include_group_specific=False` sets them to zero rather than
averaging over new sites.

## Dose-response

A dose-response curve standardizes over the same reference rows at each dose. We stay
inside the observed doses, since the model has no support elsewhere. Passing
`reference_value` turns the curve into contrasts against that dose.

In [ ]:
dose_response = g_computation(
    model, idata, estimand="dose_response", values=[0.0, 0.5, 1.0], **causal_inputs
)
plot_effects(dose_response, main="value")

Reference weights change the target population. For example, `weights` that are three
times larger for `z = 0` give an ATE for a population with more people in that stratum.
They are normalized within each selected population and are not propensity scores.

## A binary outcome: risk difference, risk ratio and odds ratio

For a Bernoulli model, the expected outcome is the probability of the event. We simulate
a conversion outcome that depends on a binary `exposure`.

In [ ]:
nb = 160
xb = rng.normal(size=nb)
zb = rng.integers(0, 2, size=nb)
exposure = rng.binomial(1, expit(-0.2 + 0.4 * xb), size=nb)
p = expit(-0.7 + 0.8 * exposure + 0.5 * xb + 0.7 * zb)
converted = np.where(rng.binomial(1, p), "converted", "not_converted")
binary_data = pd.DataFrame(
    {
        "converted": pd.Categorical(converted, categories=["converted", "not_converted"]),
        "exposure": exposure,
        "x": xb,
        "z": zb,
    }
)

In [ ]:
binary_model = bmb.Model("converted ~ exposure + x + z", binary_data, family="bernoulli")
binary_idata = binary_model.fit(random_seed=105502)

Bambi models the probability of `converted`. That is the default event for
`g_computation`; pass `event="not_converted"` to work with the complementary risk.

The risk ratio is $M_s(1) / M_s(0)$ and the marginal odds ratio is
$\frac{M_s(1) / (1 - M_s(1))}{M_s(0) / (1 - M_s(0))}$, both computed from standardized risks
within each draw. The marginal odds ratio generally differs from the exponentiated
`exposure` coefficient, even without confounding, because odds ratios are not
collapsible.

In [ ]:
binary_inputs = {
    "treatment": "exposure",
    "outcome": "converted",
    "adjustment_set": ["x", "z"],
    "temporal_order": ["x", "z", "exposure", "converted"],
    "reference_population": binary_data[["exposure", "x", "z"]],
    "values": [0, 1],
}

results = [
    g_computation(
        binary_model, binary_idata, estimand="ate", comparison=comparison, **binary_inputs
    )
    for comparison in ["difference", "risk_ratio", "odds_ratio"]
]
pd.concat([result.summary for result in results])

In [ ]:
plot_effects(results[1], main="estimand")

The dashed line marks no effect: 0 for differences and 1 for ratios.

Supported families are Gaussian, Beta, Gamma, Weibull, Exponential, LogNormal, ExGaussian
and Bernoulli. For LogNormal the expected outcome is $\exp(\mu + \sigma^2 / 2)$ and for
ExGaussian it is $\mu + \nu$, so `g_computation` uses every predicted parameter, not only
the location. Other families, and truncated, censored or transformed responses, raise an
error instead of returning an effect on the wrong scale.

## Summary

`g_computation` turns a fitted Bambi model into posterior causal effects by predicting
expected outcomes for a fixed reference population under each intervention. The
intervals reflect posterior uncertainty given that population. When reporting results,
state the interventions, the reference population and weights, the adjustment set, and
why the identification assumptions are plausible.

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w